# cuecard — Step-by-Step Pipeline Visualization

Interactive exploration of each pipeline step: parse, embed, index, retrieve, format.
Plus evaluation metrics and model comparison.

**D17 Rule:** Zero pipeline logic in this notebook. All computation delegates to `cuecard` imports.

In [ ]:
import sys
from pathlib import Path

# Ensure cuecard is importable
project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import numpy as np
import matplotlib.pyplot as plt

print(f"Project root: {project_root}")

## Step 1: Parse Rules

Load and display all rules from a corpus file with provenance.

In [ ]:
from cuecard.parser import parse_rules

CORPUS_PATH = str(project_root / "eval" / "corpora" / "rules_basic.txt")
rules = parse_rules((CORPUS_PATH,))

print(f"Parsed {len(rules)} rules from {CORPUS_PATH}\n")
for i, rule in enumerate(rules, 1):
    print(f"  {i:3d}  {rule.text}")
    print(f"       {rule.provenance.file}:{rule.provenance.line_start}")

### Rule Length Distribution

In [ ]:
lengths = [len(r.text) for r in rules]
word_counts = [len(r.text.split()) for r in rules]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(lengths, bins=20, edgecolor="black", alpha=0.7)
ax1.set_xlabel("Character count")
ax1.set_ylabel("Number of rules")
ax1.set_title("Rule Length Distribution (chars)")
ax1.axvline(np.mean(lengths), color="red", linestyle="--", label=f"mean={np.mean(lengths):.0f}")
ax1.legend()

ax2.hist(word_counts, bins=15, edgecolor="black", alpha=0.7, color="orange")
ax2.set_xlabel("Word count")
ax2.set_ylabel("Number of rules")
ax2.set_title("Rule Length Distribution (words)")
ax2.axvline(np.mean(word_counts), color="red", linestyle="--", label=f"mean={np.mean(word_counts):.1f}")
ax2.legend()

plt.tight_layout()
plt.show()

## Step 2: Embed Rules

Encode rules using the embedding model (asymmetric: `passage_embed()`).

In [ ]:
from fastembed import TextEmbedding
from cuecard.indexer import build_index
from cuecard.freshness import check_freshness

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = TextEmbedding(model_name=MODEL_NAME)

freshness = check_freshness((CORPUS_PATH,), {})
index = build_index(
    tuple(rules),
    freshness.updated_sources,
    MODEL_NAME,
    model=model,
)

print(f"Index: {index}")
print(f"Embeddings shape: {index.embeddings.shape}")
print(f"L2 norms (should be ~1.0): {np.linalg.norm(index.embeddings, axis=1)[:5]}")

### Embedding Similarity Heatmap

All-pairs cosine similarity between rules.

In [ ]:
sim_matrix = index.embeddings @ index.embeddings.T

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="RdYlBu_r", vmin=0, vmax=1)
ax.set_title("Rule-Rule Cosine Similarity")
ax.set_xlabel("Rule index")
ax.set_ylabel("Rule index")
plt.colorbar(im, ax=ax, shrink=0.8)

# Annotate with short rule labels
short_labels = [r.text[:30] + "..." if len(r.text) > 30 else r.text for r in rules]
ax.set_xticks(range(len(rules)))
ax.set_yticks(range(len(rules)))
ax.set_xticklabels(range(1, len(rules) + 1), fontsize=7)
ax.set_yticklabels([f"{i+1}: {l}" for i, l in enumerate(short_labels)], fontsize=6)

plt.tight_layout()
plt.show()

### t-SNE Visualization of Rule Embeddings

In [ ]:
from sklearn.manifold import TSNE

perplexity = min(5, len(rules) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
embeddings_2d = tsne.fit_transform(index.embeddings)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=80, alpha=0.7)

for i, rule in enumerate(rules):
    label = rule.text[:40] + "..." if len(rule.text) > 40 else rule.text
    ax.annotate(label, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=6, alpha=0.8, ha="left", va="bottom")

ax.set_title(f"t-SNE of Rule Embeddings ({MODEL_NAME})")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

## Step 3: Retrieve

Run queries against the index (asymmetric: `query_embed()`).

In [ ]:
from cuecard.retriever import retrieve
from cuecard.formatter import format_rules, format_rules_verbose

queries = [
    "Bash: git commit -m 'fix auth bug'",
    "Bash: tmux send-keys -t cody 'hello'",
    "Bash: pip install requests",
    "Edit: src/auth.py: def login(user_input)",
    "Bash: python -c 'eval(input())'",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(index, query, model=model, threshold=0.2, top_k=5)
    print(format_rules_verbose(results))

### Score Distribution Across All Fixtures

In [ ]:
import json

FIXTURES_PATH = str(project_root / "eval" / "fixtures" / "basic.json")
with open(FIXTURES_PATH) as f:
    fixtures_raw = json.load(f)

all_scores = []
match_scores = []
non_match_scores = []

for fixture in fixtures_raw:
    results = retrieve(index, fixture["query"], model=model, threshold=0.0, top_k=50)
    should_match = set(fixture["should_match"])
    should_not_match = set(fixture["should_not_match"])
    
    for r in results:
        all_scores.append(r.score)
        if r.rule.text in should_match:
            match_scores.append(r.score)
        elif r.rule.text in should_not_match:
            non_match_scores.append(r.score)

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 40)
ax.hist(all_scores, bins=bins, alpha=0.4, label=f"All ({len(all_scores)})", color="gray")
ax.hist(match_scores, bins=bins, alpha=0.7, label=f"should_match ({len(match_scores)})", color="green")
ax.hist(non_match_scores, bins=bins, alpha=0.7, label=f"should_not_match ({len(non_match_scores)})", color="red")

# Default threshold
ax.axvline(0.35, color="blue", linestyle="--", linewidth=2, label="threshold=0.35")

ax.set_xlabel("Cosine Similarity Score")
ax.set_ylabel("Count")
ax.set_title("Score Distribution: Relevant vs Irrelevant Rules")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nShould-match scores: mean={np.mean(match_scores):.3f}, min={np.min(match_scores):.3f}, max={np.max(match_scores):.3f}")
if non_match_scores:
    print(f"Should-not-match scores: mean={np.mean(non_match_scores):.3f}, min={np.min(non_match_scores):.3f}, max={np.max(non_match_scores):.3f}")
else:
    print("Should-not-match: none retrieved (good!)")

## Step 4: Format

Show the final injectable text that gets injected into agent context.

In [ ]:
sample_query = "Bash: git commit -m 'fix auth bug'"
results = retrieve(index, sample_query, model=model, top_k=5)

print("Formatted output (injected into additionalContext):")
print("---")
print(format_rules(results))
print("---")

## Step 5: Evaluation

Run the full evaluation framework on golden fixtures.

In [ ]:
from cuecard.eval import load_fixtures, run_eval, format_eval_report

fixtures = load_fixtures(FIXTURES_PATH)
corpus_dir = str(project_root / "eval" / "corpora")

summary = run_eval(
    fixtures,
    corpus_dir=corpus_dir,
    model_name=MODEL_NAME,
    model=model,
    top_k=5,
    threshold=0.35,
)

print(format_eval_report(summary))

### Per-Fixture Metrics Visualization

In [ ]:
fixture_ids = [r.fixture_id for r in summary.per_fixture]
precisions = [r.precision_at_k for r in summary.per_fixture]
recalls = [r.recall_at_k for r in summary.per_fixture]
mrrs = [r.mrr for r in summary.per_fixture]
ndcgs = [r.ndcg_at_k for r in summary.per_fixture]

x = np.arange(len(fixture_ids))
width = 0.2

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - 1.5*width, precisions, width, label="Precision@k", alpha=0.8)
ax.bar(x - 0.5*width, recalls, width, label="Recall@k", alpha=0.8)
ax.bar(x + 0.5*width, mrrs, width, label="MRR", alpha=0.8)
ax.bar(x + 1.5*width, ndcgs, width, label="nDCG@k", alpha=0.8)

ax.set_xlabel("Fixture")
ax.set_ylabel("Score")
ax.set_title(f"Per-Fixture IR Metrics ({MODEL_NAME})")
ax.set_xticks(x)
ax.set_xticklabels(fixture_ids, rotation=45, ha="right", fontsize=7)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### Latency Distribution

In [ ]:
latencies = [r.latency_ms for r in summary.per_fixture]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(fixture_ids, latencies, alpha=0.7, color="steelblue")
ax.axhline(summary.latency_p50_ms, color="green", linestyle="--", label=f"p50={summary.latency_p50_ms:.1f}ms")
ax.axhline(summary.latency_p95_ms, color="orange", linestyle="--", label=f"p95={summary.latency_p95_ms:.1f}ms")
ax.axhline(summary.latency_p99_ms, color="red", linestyle="--", label=f"p99={summary.latency_p99_ms:.1f}ms")

ax.set_xlabel("Fixture")
ax.set_ylabel("Latency (ms)")
ax.set_title("Retrieval Latency per Fixture")
ax.set_xticklabels(fixture_ids, rotation=45, ha="right", fontsize=7)
ax.legend()
plt.tight_layout()
plt.show()

### Aggregate Summary

In [ ]:
print(f"{'Metric':<25} {'Value':>10}")
print(f"{'-'*25} {'-'*10}")
print(f"{'Fixtures':<25} {summary.fixture_count:>10d}")
print(f"{'Mean Precision@k':<25} {summary.mean_precision:>10.3f}")
print(f"{'Mean Recall@k':<25} {summary.mean_recall:>10.3f}")
print(f"{'Mean MRR':<25} {summary.mean_mrr:>10.3f}")
print(f"{'Mean nDCG@k':<25} {summary.mean_ndcg:>10.3f}")
print(f"{'Mean Anti-Precision':<25} {summary.mean_anti_precision:>10.3f}")
print(f"{'Latency p50 (ms)':<25} {summary.latency_p50_ms:>10.1f}")
print(f"{'Latency p95 (ms)':<25} {summary.latency_p95_ms:>10.1f}")
print(f"{'Latency p99 (ms)':<25} {summary.latency_p99_ms:>10.1f}")

## Step 6: Model Comparison

Compare different embedding models on the same fixtures.

**Note:** Each model must be downloaded first via `cuecard setup --model <name>`.

In [ ]:
# Configure models to compare (uncomment those you've downloaded)
MODELS_TO_COMPARE = [
    "BAAI/bge-small-en-v1.5",
    # "nomic-ai/nomic-embed-text-v1.5",
    # "snowflake/snowflake-arctic-embed-m",
    # "jinaai/jina-embeddings-v2-base-code",
    # "mixedbread-ai/mxbai-embed-large-v1",
]

comparison_results = {}
for model_name in MODELS_TO_COMPARE:
    print(f"\nEvaluating: {model_name}")
    m = TextEmbedding(model_name=model_name)
    s = run_eval(fixtures, corpus_dir=corpus_dir, model_name=model_name, model=m)
    comparison_results[model_name] = s
    print(f"  Precision={s.mean_precision:.3f}  Recall={s.mean_recall:.3f}  "
          f"MRR={s.mean_mrr:.3f}  nDCG={s.mean_ndcg:.3f}  "
          f"AntiPrec={s.mean_anti_precision:.3f}  p50={s.latency_p50_ms:.1f}ms")

In [ ]:
if len(comparison_results) > 1:
    model_names = list(comparison_results.keys())
    short_names = [n.split("/")[-1] for n in model_names]
    metrics = ["mean_precision", "mean_recall", "mean_mrr", "mean_ndcg"]
    metric_labels = ["Precision@k", "Recall@k", "MRR", "nDCG@k"]

    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)

    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (name, short) in enumerate(zip(model_names, short_names)):
        s = comparison_results[name]
        values = [getattr(s, m) for m in metrics]
        offset = (i - len(model_names)/2 + 0.5) * width
        ax.bar(x + offset, values, width, label=short, alpha=0.8)

    ax.set_ylabel("Score")
    ax.set_title("Model Comparison on Golden Fixtures")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.legend()
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.show()
else:
    print("Only one model evaluated. Uncomment more models above to compare.")

## Step 7: Threshold Sensitivity

How do metrics change across different threshold values?

In [ ]:
thresholds = np.arange(0.0, 0.8, 0.05)
threshold_precisions = []
threshold_recalls = []
threshold_anti = []

for t in thresholds:
    s = run_eval(fixtures, corpus_dir=corpus_dir, model_name=MODEL_NAME,
                 model=model, threshold=float(t))
    threshold_precisions.append(s.mean_precision)
    threshold_recalls.append(s.mean_recall)
    threshold_anti.append(s.mean_anti_precision)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, threshold_precisions, "o-", label="Precision@k", linewidth=2)
ax.plot(thresholds, threshold_recalls, "s-", label="Recall@k", linewidth=2)
ax.plot(thresholds, threshold_anti, "^-", label="Anti-Precision", linewidth=2, color="red")
ax.axvline(0.35, color="gray", linestyle="--", alpha=0.5, label="default=0.35")

ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Sensitivity Analysis")
ax.legend()
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()